# Laptop Market Analysis - URL Harvesting

## 1. Import Libraries

In [ ]:
import time
import json
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

## 2. Site Configuration Profiles

In [ ]:
# tell the crawler the starting url, what is the structure of the popup, laptop links, "See more" button
SITE_CONFIGS = {
    "phongvu": {
        "source_name": "Phong Vu",
        "base_url": "https://phongvu.vn",
        "start_url": "https://phongvu.vn/c/laptop",
        "link_keywords": ["laptop", "macbook"],
        "exclude_keywords": [],
        "load_more_xpath": "//button[contains(., 'Xem thêm')] | //div[contains(text(), 'Xem thêm')]",
        "popup_selectors": [
            "button#onetrust-accept-btn-handler",   
            "div.css-1n790y4",                      
            "button.css-1v77y0u",                   
            "span.close-btn",
            "button[aria-label='Close']"
        ]
    },
    
    "cellphones": {
        "source_name": "CellphoneS",
        "base_url": "https://cellphones.com.vn",
        "start_url": "https://cellphones.com.vn/laptop.html",
        "link_keywords": ["laptop", "macbook"],
        "exclude_keywords": [],
        "load_more_xpath": "//a[contains(@class, 'btn-show-more')]",
        "popup_selectors": [
            ".btn-confirm-address",                 
            "#onesignal-slidedown-cancel-button",   
            ".cancel-button-top",                   
            ".close-modal"                          
        ]
    },
    
    "tgdd": {
        "source_name": "The Gioi Di Dong",
        "base_url": "https://www.thegioididong.com",
        "start_url": "https://www.thegioididong.com/laptop",
        "link_keywords": ["laptop", "macbook"],
        "exclude_keywords": ["balo", "chuot", "cap", "phu-kien", "tui-chong-soc", "phan-mem"],
        "load_more_xpath": "//div[contains(@class, 'view-more')]//a | //a[contains(text(), 'Xem thêm')]",
        "popup_selectors": [
            ".close-popup",                        
            "#onesignal-slidedown-cancel-button"
        ]
    },

    "fpt": {
        "source_name": "FPT Shop",
        "base_url": "https://fptshop.com.vn/",
        "start_url": "https://fptshop.com.vn/may-tinh-xach-tay",
        "link_keywords": ["laptop", "macbook", "may-tinh-xach-tay"],
        "exclude_keywords": ["balo", "chuot", "cap", "phu-kien", "tui-chong-soc", "phan-mem"],
        "load_more_xpath": "//button[contains(normalize-space(.), 'Xem thêm') and contains(normalize-space(.), 'kết quả')]",
        "popup_selectors": [
            ".close-popup",                         
            "#onesignal-slidedown-cancel-button"
        ]
    },

    "gearvn": {
        "source_name": "Gear VN",
        "base_url": "https://gearvn.com",
        "start_url": "https://gearvn.com/collections/laptop",
        "link_keywords": ["/products/"],  
        "exclude_keywords": ["balo", "chuot", "tui-chong-soc", "lot-chuot", "phan-mem"],
        "load_more_xpath": "//button[@id='load_more']",
        "popup_selectors": [
            "#popup-banner .close-popup",          
            ".modal-promotion .close",              
            "#onesignal-slidedown-cancel-button",   
            ".close-modal",                         
            "button.btn-close"                      
        ]
    },

    "anphat": {
        "source_name": "An Phat PC",
        "base_url": "https://www.anphatpc.com.vn",
        "start_url": "https://www.anphatpc.com.vn/may-tinh-xach-tay-laptop.html",
        "link_keywords": ["/laptop", "macbook"],
        "exclude_keywords": ["balo", "chuot", "tui-chong-soc", "lot-chuot", "voucher", "phan-mem"],
        "load_more_xpath": "//a[@id='js-product-count'] | //a[contains(@class, 'btn-view-more')]",
        "popup_selectors": [
            ".global-banner-popup-holder .fa-times-circle", 
            ".bg-popup",                                    
            ".close-modal",
            "#onesignal-slidedown-cancel-button"
        ]
    },

    "thinkpro": {
        "source_name": "Think Pro",
        "base_url": "https://thinkpro.vn",
        "start_url": "https://thinkpro.vn/laptop",
        "link_keywords": ["/laptop/"],
        "exclude_keywords": ["balo", "chuot", "tui-chong-soc", "lot-chuot", "voucher", "phan-mem"],
        "load_more_xpath": "//button[contains(., 'Xem thêm')] | //div[contains(text(), 'Xem thêm')]",
        "popup_selectors": [
            ".global-banner-popup-holder .fa-times-circle", 
            ".bg-popup",                                    
            ".close-modal",
            "#onesignal-slidedown-cancel-button"
        ]
    }
}


## 3. Crawler Implementation
The `Crawler` class handles browser initialization, ad dismissal, and link extraction.

**Harvesting Approach:**
1. **Handle Ads:** Detect and dismiss pop-ups or advertisements.
2. **Navigate:** Go to the specific laptop category URL.
3. **Extract:** Scrape `<a>` tags matching `link_keywords` while ignoring `exclude_keywords`.
4. **Load More:** Locate the "Xem thêm" button, scroll into view, and click to trigger lazy loading.
5. **Repeat:** Continue the loop until no new links are found or the button is hidden.

In [1]:
# the crawling process is the same for all websites: 
# - get to the laptop section of the webpage
# - dismiss any popups
# - save laptop urls to a set, scroll down, press "See more" and save new links 
# - repeat until we cant press "See more" anymore

class Crawler:
    def __init__(self, headless=False):
        options = webdriver.ChromeOptions()
        if headless:
            options.add_argument('--headless') 
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')
        options.add_argument('--no-sandbox')
        
        self.driver = webdriver.Chrome(options=options)

        self.driver.set_page_load_timeout(30)

        self.wait = WebDriverWait(self.driver, 5) 

    # dismiss any popup ads
    def handle_popups(self, popup_selectors):
        for selector in popup_selectors:
            try:
                elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                for el in elements:
                    if el.is_displayed():
                        self.driver.execute_script("arguments[0].click();", el)
                        print(f"  [+] Dismissed popup: {selector}")
                        time.sleep(1)
            except Exception:
                pass 

    def harvest_urls(self, config):
        print(f"Started crawling for: {config['source_name']}")
        
        try:
            self.driver.get(config['start_url'])
        except TimeoutException:
            print("- Timeout")

        time.sleep(3) 
        
        self.handle_popups(config['popup_selectors'])
        collected_urls = set()

        click_attempt = 0     
        MAX_STAGNANT_CLICKS = 3  # stop if we click see more 3 times and still find no new links
        
        while True:
            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            all_links = soup.find_all("a", href=True)
            
            new_finds = 0
            for a in all_links:
                href = a.get('href').lower()
                
                # check if link follows laptop link structure
                has_keyword = any(kw in href for kw in config['link_keywords'])

                # check if link has any forbidden words
                has_exclusion = any(ex in href for ex in config['exclude_keywords'])
                
                if has_keyword and not has_exclusion:
                    full_url = a.get('href') if a.get('href').startswith("http") else f"{config['base_url']}{a.get('href')}"
                    
                    if full_url not in collected_urls:
                        collected_urls.add(full_url)
                        new_finds += 1

            if new_finds == 0 and len(collected_urls) > 0:
                click_attempt += 1
                print(f"- No new links loaded. Attempt {click_attempt}/{MAX_STAGNANT_CLICKS}")
                if click_attempt >= MAX_STAGNANT_CLICKS:
                    print("- Timeout")
                    break
            else:
                click_attempt = 0 


            # find the "see more" button
            try:
                load_more_btn = self.driver.find_element(By.XPATH, config['load_more_xpath'])
                
                if not load_more_btn.is_displayed():
                    print("- Button hidden. Checking for popups")
                    self.handle_popups(config['popup_selectors'])
                    time.sleep(1)
                    
                    if not load_more_btn.is_displayed():
                        print("-- Button is still hidden. Reached the end of the website")
                        break
                
                self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
                time.sleep(1)
                self.driver.execute_script("arguments[0].click();", load_more_btn)
                
                print(f"  [>] Clicked 'See More'. Total unique URLs: {len(collected_urls)} (+{new_finds} new)")
                time.sleep(2) 
                
            except NoSuchElementException:
                print("- No 'Xem thêm' button found, reached the end.")
                break
            except Exception as e:
                print(f"- Error, loop ended : {e}")
                break

        results = []
        for url in collected_urls:
            name_guess = url.split('/')[-1].replace('.html', '').replace('-', ' ').title()
            
            results.append({
                "source": config['source_name'],
                "name": name_guess,
                "url": url
            })
            
        print(f"Extracted {len(results)} links from {config['source_name']}.\n")
        return results

    def save_json(self, data, filename):
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"Saved {len(data)} items to {filename}")

    def close(self):
        self.driver.quit()

## 4. Data Collection

In [ ]:
crawler = Crawler(headless=False)
    
try:
    pv_data = crawler.harvest_urls(SITE_CONFIGS["phongvu"])
    crawler.save_json(pv_data, "urls/01_phongvu_urls.json")
finally:
    crawler.close()

--- Starting Harvest for: Phong Vu ---
  [>] Clicked 'Xem thêm'. Total unique URLs: 158 (+158 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 195 (+37 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 231 (+36 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 269 (+38 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 305 (+36 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 341 (+36 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 380 (+39 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 419 (+39 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 459 (+40 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 463 (+4 new)
  [-] No 'Xem thêm' button found in DOM. Reached the end.
✅ Success! Extracted 464 laptops from Phong Vu.

Saved 464 items to urls/01_phongvu_urls.json


In [ ]:
crawler = Crawler(headless=False)
    
try:
    cps_data = crawler.harvest_urls(SITE_CONFIGS["cellphones"])
    crawler.save_json(cps_data, "urls/01_cellphones_urls.json")
finally:
    crawler.close()

--- Starting Harvest for: CellphoneS ---
  [>] Clicked 'Xem thêm'. Total unique URLs: 102 (+102 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 119 (+17 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 135 (+16 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 150 (+15 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 167 (+17 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 185 (+18 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 204 (+19 new)
  [*] 'Xem thêm' button hidden. Checking for popups/ads...
  [+] Dismissed popup: .cancel-button-top
  [>] Clicked 'Xem thêm'. Total unique URLs: 224 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 243 (+19 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 262 (+19 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 281 (+19 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 301 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 321 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 339 (+18 new)
  [>] Clicked 'Xem thêm'. Total

In [ ]:
crawler = Crawler(headless=False)

try:
    tgdd_data = crawler.harvest_urls(SITE_CONFIGS["tgdd"])
    crawler.save_json(tgdd_data, "urls/01_tgdd_urls.json")
finally:
    crawler.close()

--- Starting Harvest for: The Gioi Di Dong ---
  [>] Clicked 'Xem thêm'. Total unique URLs: 168 (+168 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 188 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 208 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 228 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 248 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 268 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 288 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 308 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 328 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 348 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 368 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 388 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 408 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 428 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 448 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 468 (+20 new)
  [>] Cl

In [ ]:
crawler = Crawler(headless=False)
    
try:
    tgdd_data = crawler.harvest_urls(SITE_CONFIGS["fpt"])
    crawler.save_json(tgdd_data, "urls/01_fpt_urls.json")
finally:
    crawler.close()

--- Starting Harvest for: FPT Shop ---
  [>] Clicked 'Xem thêm'. Total unique URLs: 105 (+105 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 129 (+24 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 153 (+24 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 174 (+21 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 183 (+9 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 203 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 216 (+13 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 233 (+17 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 257 (+24 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 277 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 293 (+16 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 314 (+21 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 334 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 348 (+14 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 372 (+24 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 396 (+24 new)
  [>] Clicked 'Xe

In [ ]:
crawler = Crawler(headless=False)

try:
    tgdd_data = crawler.harvest_urls(SITE_CONFIGS["gearvn"])
    crawler.save_json(tgdd_data, "urls/01_gearvn_urls.json")
finally:
    crawler.close()

--- Starting Harvest for: Gear VN ---
  [>] Clicked 'Xem thêm'. Total unique URLs: 62 (+62 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 112 (+50 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 162 (+50 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 212 (+50 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 262 (+50 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 309 (+47 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 359 (+50 new)
  [*] 'Xem thêm' button hidden. Checking for popups/ads...
  [-] 'Xem thêm' button is still hidden. Reached the end of the list.
✅ Success! Extracted 363 laptops from Gear VN.

Saved 363 items to urls/01_gearvn_urls.json


In [ ]:
crawler = Crawler(headless=False)

try:
    tgdd_data = crawler.harvest_urls(SITE_CONFIGS["anphat"])
    crawler.save_json(tgdd_data, "urls/01_anphat_urls.json")
finally:
    crawler.close()

--- Starting Harvest for: An Phat PC ---
  [>] Clicked 'Xem thêm'. Total unique URLs: 162 (+162 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 190 (+28 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 219 (+29 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 247 (+28 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 275 (+28 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 304 (+29 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 333 (+29 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 362 (+29 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 392 (+30 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 421 (+29 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 451 (+30 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 480 (+29 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 510 (+30 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 540 (+30 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 570 (+30 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 600 (+30 new)
  [>] Clicked 

In [ ]:
crawler = Crawler(headless=False)

try:
    tgdd_data = crawler.harvest_urls(SITE_CONFIGS["thinkpro"])
    crawler.save_json(tgdd_data, "urls/01_thinkpro_urls.json")
finally:
    crawler.close()

--- Starting Harvest for: Think Pro ---
  [>] Clicked 'Xem thêm'. Total unique URLs: 137 (+137 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 157 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 176 (+19 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 196 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 216 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 236 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 251 (+15 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 251 (+0 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 257 (+6 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 269 (+12 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 289 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 308 (+19 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 328 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 337 (+9 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 356 (+19 new)
  [-] No 'Xem thêm' button found in DOM. Reached the end.
✅ Success! Extracte

In [ ]:
crawler = Crawler(headless=False)

try:
    tgdd_data = crawler.harvest_urls(SITE_CONFIGS["thinkpro"])
    crawler.save_json(tgdd_data, "urls/02_thinkpro_urls.json")
finally:
    crawler.close()

--- Starting Harvest for: Think Pro ---
  [>] Clicked 'Xem thêm'. Total unique URLs: 137 (+137 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 157 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 177 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 197 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 217 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 237 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 257 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 277 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 297 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 317 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 336 (+19 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 356 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 376 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 396 (+20 new)
  [>] Clicked 'Xem thêm'. Total unique URLs: 414 (+18 new)
  [-] No 'Xem thêm' button found in DOM. Reached the end.
✅ Success! Extra